AI Assistance: OpenAI ChatGPT and Anthropic's Claude were used for code debugging, code generation, code organization, and code methodological brainstorming. All final modeling, implementation, validation, commentary, and interpretation were performed and verified by the authors.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

platform = 'deepnote'

if platform == 'deepnote':
    PROCESSED_DIR = Path('data/Processed')
if platform == 'vscode':
    PROCESSED_DIR = Path('work/Processed')

# I. Function definitions

## 1. `resample_to_hourly`

In [2]:
RAW_VARS = [
    'mag_avg_nt',
    'bx_gsm_nt',
    'by_gsm_nt',
    'bz_gsm_nt',
    'flow_speed_km_s',
    'proton_density_n_cc',
]

TIME_ENCODING_COLS = ['hour_sin', 'hour_cos', 'day_sin', 'day_cos']


def resample_to_hourly(df_minute):
    """
    Resample per-minute OMNI data to hourly aggregates for LSTM feature building.

    Computes hourly mean/min/max for the 6 raw solar-wind variables, carries
    through the hour/day cyclical time encodings via mean (they are constant
    within their period so this is a lossless pass-through, not a
    recomputation), computes the kp_index target (kp_10 / 10), and records a
    per-hour minute count used downstream by drop_incomplete_intervals for
    missingness QC.

    Parameters
    ----------
    df_minute : pd.DataFrame
        Per-minute dataframe for a single split (train or test). Must contain
        'datetime', the 6 raw solar-wind columns, the 4 time-encoding columns,
        and 'kp_10'.

    Returns
    -------
    pd.DataFrame
        Hourly-indexed dataframe covering the full date range, with 18 raw
        stat columns, 4 time-encoding columns, 'kp_index', and 'minute_count'.
    """
    print('--- resample_to_hourly: record count audit ---')
    print(f'Input rows (per-minute): {len(df_minute):,}')

    # Index by datetime so resample() can bucket rows by calendar hour
    d = df_minute.set_index('datetime').sort_index()

    # 18 raw channels: mean/min/max per solar-wind variable
    raw_stats = d[RAW_VARS].resample('1h').agg(['mean', 'min', 'max'])
    raw_stats.columns = [f'{col}_{stat}' for col, stat in raw_stats.columns]

    # 4 time-encoding channels - constant within the hour/day, so mean just
    # carries the existing per-minute value through untouched
    time_encodings = d[TIME_ENCODING_COLS].resample('1h').mean()

    # Target: hourly mean of kp_10 (constant across a 3-hour block, so this
    # equals the instantaneous value whenever the hour has any data at all),
    # converted from Kp*10 to the actual Kp index
    kp_index = d['kp_10'].resample('1h').mean() / 10.0

    # Minute counts per hour - needed by drop_incomplete_intervals to measure
    # missingness; an entirely absent hour resamples to a count of 0, not NaN
    minute_count = d.resample('1h').size()

    hourly = pd.concat([raw_stats, time_encodings], axis=1)
    hourly['kp_index'] = kp_index
    hourly['minute_count'] = minute_count

    print(f'Resampled hourly buckets ({hourly.index.min()} to {hourly.index.max()}): {len(hourly):,}')
    print(f'Output rows (hourly): {len(hourly):,}')

    return hourly

## 2. `drop_incomplete_intervals`

In [3]:
def drop_incomplete_intervals(df_hourly, max_missing_minutes=5):
    """
    Identify valid 24-hour lookback intervals for each 3-hour Kp boundary hour
    and drop any whose lookback window or target is incomplete.

    For each candidate boundary hour H (H % 3 == 0), this requires a full,
    in-range 24-hour history before H (H-24h ... H-1h) with no single hour in
    that window missing more than `max_missing_minutes` minutes of underlying
    per-minute data, plus a non-null kp_index target at H itself. Failing
    intervals are dropped entirely (not truncated), so every kept interval
    yields a full, fixed-length 24-hour sequence downstream - no padding or
    masking is ever needed later in the pipeline.

    Parameters
    ----------
    df_hourly : pd.DataFrame
        Output of resample_to_hourly - must contain 'kp_index' and
        'minute_count'.
    max_missing_minutes : int, default 5
        Maximum allowed missing minutes in any single lookback hour.

    Returns
    -------
    kept_intervals_df : pd.DataFrame
        Indexed by target hour H, with the kp_index target for each kept
        interval.
    dropped_intervals_df : pd.DataFrame
        Indexed by target hour H, with boolean reason columns for each
        dropped candidate.
    """
    print('--- drop_incomplete_intervals: record count audit ---')
    print(f'Input hourly rows: {len(df_hourly):,}')

    # Candidate boundary hours - Kp is officially reported on a 3-hour cadence
    candidates = df_hourly.index[df_hourly.index.hour % 3 == 0]
    print(f'Candidate 3-hour boundary hours (hour % 3 == 0): {len(candidates):,}')

    missing_minutes = 60 - df_hourly['minute_count']

    # Max missing-minutes among the 24 hours strictly preceding each hour.
    # rolling(24) covers rows [i-23, i]; shifting by 1 re-aligns that value to
    # row i+1, i.e. the 24 hours (i+1)-24 .. (i+1)-1 preceding row i+1. Rows
    # without 24 full prior hours (start of the series) come out as NaN.
    window_max_missing = missing_minutes.rolling(window=24).max().shift(1)

    candidate_window_missing = window_max_missing.loc[candidates]
    candidate_kp_index = df_hourly['kp_index'].loc[candidates]

    no_full_history = candidate_window_missing.isna()
    eligible = candidates[~no_full_history]
    print(f'Eligible candidates with a full 24-hour history in range: {len(eligible):,}')
    print(f'  - dropped for insufficient history (start of series): {int(no_full_history.sum()):,}')

    fail_window = candidate_window_missing.loc[eligible] > max_missing_minutes
    fail_target = candidate_kp_index.loc[eligible].isna()
    drop_mask = (fail_window | fail_target).to_numpy()

    kept = eligible[~drop_mask]

    print(f'  - dropped for >{max_missing_minutes} missing minutes in the lookback window: '
          f'{int(fail_window.sum()):,}')
    print(f'  - dropped for a missing target kp_index: {int(fail_target.sum()):,}')

    kept_intervals_df = pd.DataFrame({'kp_index': candidate_kp_index.loc[kept]})
    kept_intervals_df.index.name = 'datetime'

    # Build one combined reason table covering all three failure types
    dropped_reasons = pd.DataFrame(index=candidates)
    dropped_reasons.index.name = 'datetime'
    dropped_reasons['insufficient_history'] = no_full_history
    dropped_reasons['missing_window'] = False
    dropped_reasons['missing_target'] = False
    dropped_reasons.loc[eligible, 'missing_window'] = fail_window.to_numpy()
    dropped_reasons.loc[eligible, 'missing_target'] = fail_target.to_numpy()
    reason_cols = ['insufficient_history', 'missing_window', 'missing_target']
    dropped_intervals_df = dropped_reasons.loc[dropped_reasons[reason_cols].any(axis=1)]

    total_dropped = len(candidates) - len(kept_intervals_df)
    print(f'Dropping {total_dropped:,} intervals ({total_dropped / len(candidates) * 100:.2f}%)')
    print(f'Keeping {len(kept_intervals_df):,} intervals')

    return kept_intervals_df, dropped_intervals_df

## 3. `build_lstm_sequences`

In [4]:
CHANNELS = [f'{var}_{stat}' for var in RAW_VARS for stat in ('mean', 'min', 'max')] + TIME_ENCODING_COLS


def build_lstm_sequences(df_hourly, kept_intervals_df, output_path):
    """
    Assemble fixed-length (24, 22) LSTM input sequences for every kept
    interval and save them to disk as .npy arrays.

    For each target hour H in kept_intervals_df, slices the 24 hourly rows
    strictly preceding H (H-24h ... H-1h) from df_hourly across all 22
    feature channels, in a fixed column order (CHANNELS). Purely mechanical -
    assumes drop_incomplete_intervals has already guaranteed every H is safe
    to slice, so no missingness or eligibility logic lives here.

    Parameters
    ----------
    df_hourly : pd.DataFrame
        Output of resample_to_hourly.
    kept_intervals_df : pd.DataFrame
        Output of drop_incomplete_intervals - one row per valid target hour H,
        with the kp_index target.
    output_path : str or Path
        Prefix path to write outputs to, e.g. 'work/Processed/lstm_train'
        writes lstm_train_X.npy, lstm_train_y.npy, lstm_train_timestamps.npy,
        and lstm_train_channels.json.

    Returns
    -------
    X : np.ndarray, shape (n_sequences, 24, 22)
    y : np.ndarray, shape (n_sequences,)
    timestamps : np.ndarray, shape (n_sequences,), dtype datetime64[ns]
    """
    print('--- build_lstm_sequences: assembly audit ---')

    # Build every possible 24-hour window across the whole hourly series in
    # one vectorized pass, then gather just the ones we need by position -
    # far faster than slicing per-interval with pandas .loc in a Python loop.
    channel_matrix = df_hourly[CHANNELS].to_numpy(dtype=np.float32)
    all_windows = np.lib.stride_tricks.sliding_window_view(channel_matrix, window_shape=24, axis=0)
    all_windows = np.transpose(all_windows, (0, 2, 1))  # -> (n_hours - 23, 24, n_channels)

    # window at position k covers hourly rows [k, k+23]; the target hour for
    # that window sits at position k+24, so look up k = position(H) - 24
    target_positions = df_hourly.index.get_indexer(kept_intervals_df.index)
    window_start_positions = target_positions - 24

    X = all_windows[window_start_positions]
    y = kept_intervals_df['kp_index'].to_numpy(dtype=np.float32)
    timestamps = kept_intervals_df.index.to_numpy(dtype='datetime64[ns]')

    print(f'Sequences assembled: {len(kept_intervals_df):,}')
    print(f'X.shape: {X.shape}')
    print(f'y.shape: {y.shape}')
    print(f'timestamps.shape: {timestamps.shape}')
    print(f'Channels ({len(CHANNELS)}): {CHANNELS}')

    output_path = Path(output_path)
    paths = {
        'X': output_path.parent / f'{output_path.name}_X.npy',
        'y': output_path.parent / f'{output_path.name}_y.npy',
        'timestamps': output_path.parent / f'{output_path.name}_timestamps.npy',
        'channels': output_path.parent / f'{output_path.name}_channels.json',
    }

    np.save(paths['X'], X)
    np.save(paths['y'], y)
    np.save(paths['timestamps'], timestamps)
    with open(paths['channels'], 'w') as f:
        json.dump(CHANNELS, f)

    print(f'Saved outputs: {list(paths.values())}')

    # Built-in sanity check: preview the first and last assembled sequences,
    # spread across both ends of the dataset rather than just the start
    for label, i in [('First', 0), ('Last', -1)]:
        preview = pd.DataFrame(X[i], columns=CHANNELS)
        preview.index.name = 'hour_offset (H-24 ... H-1)'
        print(f"\n--- {label} sequence preview (target hour {timestamps[i]}) ---")
        print(preview)
        print(f'  -> target kp_index = {y[i]:.2f}')

    return X, y, timestamps

# II. Run pipeline — train set

In [5]:
# Load the data
COLUMNS_NEEDED = ['datetime'] + RAW_VARS + TIME_ENCODING_COLS + ['kp_10']

df_train_minute = pd.read_parquet(PROCESSED_DIR / 'omni_train_cleaned.parquet', columns=COLUMNS_NEEDED)
df_train_minute.shape

(10334497, 12)

In [6]:
# Drop any remaining nulls
df_train_minute = df_train_minute.dropna()

In [9]:
# Re-sample train set to hourly
df_train_hourly = resample_to_hourly(df_train_minute)

--- resample_to_hourly: record count audit ---
Input rows (per-minute): 10,334,497
Resampled hourly buckets (2000-01-01 00:00:00 to 2019-12-31 23:00:00): 175,320
Output rows (hourly): 175,320


In [10]:
# Drop incomplete 24-hour intervals from the training set
kept_train, dropped_train = drop_incomplete_intervals(df_train_hourly)

--- drop_incomplete_intervals: record count audit ---
Input hourly rows: 175,320
Candidate 3-hour boundary hours (hour % 3 == 0): 58,440
Eligible candidates with a full 24-hour history in range: 58,432
  - dropped for insufficient history (start of series): 8
  - dropped for >5 missing minutes in the lookback window: 7,917
  - dropped for a missing target kp_index: 762
Dropping 7,943 intervals (13.59%)
Keeping 50,497 intervals


In [12]:
# Build the LSTM sequences for train set
X_train, y_train, ts_train = build_lstm_sequences(
    df_train_hourly, kept_train, output_path=PROCESSED_DIR / 'lstm_train'
)

--- build_lstm_sequences: assembly audit ---
Sequences assembled: 50,497
X.shape: (50497, 24, 22)
y.shape: (50497,)
timestamps.shape: (50497,)
Channels (22): ['mag_avg_nt_mean', 'mag_avg_nt_min', 'mag_avg_nt_max', 'bx_gsm_nt_mean', 'bx_gsm_nt_min', 'bx_gsm_nt_max', 'by_gsm_nt_mean', 'by_gsm_nt_min', 'by_gsm_nt_max', 'bz_gsm_nt_mean', 'bz_gsm_nt_min', 'bz_gsm_nt_max', 'flow_speed_km_s_mean', 'flow_speed_km_s_min', 'flow_speed_km_s_max', 'proton_density_n_cc_mean', 'proton_density_n_cc_min', 'proton_density_n_cc_max', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos']
Saved outputs: [WindowsPath('work/Processed/lstm_train_X.npy'), WindowsPath('work/Processed/lstm_train_y.npy'), WindowsPath('work/Processed/lstm_train_timestamps.npy'), WindowsPath('work/Processed/lstm_train_channels.json')]

--- First sequence preview (target hour 2000-01-02T00:00:00.000000000) ---
                            mag_avg_nt_mean  mag_avg_nt_min  mag_avg_nt_max  \
hour_offset (H-24 ... H-1)                          

In [13]:
# Function to validate 24 hour sequence

def validate_lstm_sequence(df_hourly, timestamps, X, channels, output_dir, seed=None):
    """
    Pick a random kept target hour and export two CSVs for manual eyeball
    comparison: the raw hourly source window it was built from, and the
    feature matrix build_lstm_sequences actually assembled for it.

    Parameters
    ----------
    df_hourly : pd.DataFrame
        Output of resample_to_hourly - the hourly source table sequences are
        sliced from.
    timestamps : np.ndarray
        timestamps array returned by build_lstm_sequences (aligned to X's
        first axis).
    X : np.ndarray, shape (n_sequences, 24, n_channels)
        X array returned by build_lstm_sequences.
    channels : list of str
        Channel names, in the same column order as X's last axis.
    output_dir : str or Path
        Directory to write the two validation CSVs to.
    seed : int, optional
        Seed for the random target-hour pick, for reproducible spot checks.

    Returns
    -------
    source_df : pd.DataFrame
        df_hourly filtered to the picked target hour and the 24 hours
        preceding it (25 rows, all columns).
    sequence_df : pd.DataFrame
        The (24, n_channels) feature matrix assembled for that target hour,
        as a labeled dataframe.
    picked_dt : pd.Timestamp
        The randomly picked target hour.
    """
    rng = np.random.default_rng(seed)
    idx = rng.integers(len(timestamps))
    picked_dt = pd.Timestamp(timestamps[idx])

    print('--- validate_lstm_sequence: spot check ---')
    print(f'Randomly picked target hour: {picked_dt}')

    # First dataframe: raw hourly source window (target hour + 24 preceding hours)
    window_start = picked_dt - pd.Timedelta(hours=24)
    source_df = df_hourly.loc[window_start:picked_dt].copy()

    # Second dataframe: the feature matrix build_lstm_sequences assembled for this target
    sequence_df = pd.DataFrame(X[idx], columns=channels)
    sequence_df.index.name = 'hour_offset (H-24 ... H-1)'

    output_dir = Path(output_dir)
    tag = picked_dt.strftime('%Y%m%dT%H%M')
    source_path = output_dir / f'lstm_validate_source_{tag}.csv'
    sequence_path = output_dir / f'lstm_validate_sequence_{tag}.csv'
    source_df.to_csv(source_path)
    sequence_df.to_csv(sequence_path)

    print(f'Source window ({len(source_df)} rows, target kp_index={source_df["kp_index"].iloc[-1]:.4f}) -> {source_path}')
    print(f'Assembled sequence {sequence_df.shape} -> {sequence_path}')

    return source_df, sequence_df, picked_dt


In [14]:
# # Export the 24-hour sequence validation reports (CSV)
# source_df, sequence_df, picked_dt = validate_lstm_sequence(
#     df_train_hourly, ts_train, X_train, CHANNELS, output_dir=PROCESSED_DIR
# )

--- validate_lstm_sequence: spot check ---
Randomly picked target hour: 2001-09-04 12:00:00
Source window (25 rows, target kp_index=2.3000) -> work\Processed\lstm_validate_source_20010904T1200.csv
Assembled sequence (24, 22) -> work\Processed\lstm_validate_sequence_20010904T1200.csv


# III. Run pipeline — validation set

In [15]:
# Load the validation set
df_validation_minute = pd.read_parquet(PROCESSED_DIR / 'omni_validation_cleaned.parquet', columns=COLUMNS_NEEDED)
df_validation_minute.shape

(1800962, 12)

In [16]:
# Drop any remaining nulls
df_validation_minute = df_validation_minute.dropna()

In [18]:
# Re-sample validation set to hourly
df_validation_hourly = resample_to_hourly(df_validation_minute)

--- resample_to_hourly: record count audit ---
Input rows (per-minute): 1,800,962
Resampled hourly buckets (2020-01-01 00:00:00 to 2023-06-30 23:00:00): 30,648
Output rows (hourly): 30,648


In [19]:
# Drop incomplete 24-hour intervals from the validation set
kept_validation, dropped_validation = drop_incomplete_intervals(df_validation_hourly)

--- drop_incomplete_intervals: record count audit ---
Input hourly rows: 30,648
Candidate 3-hour boundary hours (hour % 3 == 0): 10,216
Eligible candidates with a full 24-hour history in range: 10,208
  - dropped for insufficient history (start of series): 8
  - dropped for >5 missing minutes in the lookback window: 1,164
  - dropped for a missing target kp_index: 172
Dropping 1,175 intervals (11.50%)
Keeping 9,041 intervals


In [20]:
# Build the LSTM sequences for validation set
X_validation, y_validation, ts_validation = build_lstm_sequences(
    df_validation_hourly, kept_validation, output_path=PROCESSED_DIR / 'lstm_validation'
)

--- build_lstm_sequences: assembly audit ---
Sequences assembled: 9,041
X.shape: (9041, 24, 22)
y.shape: (9041,)
timestamps.shape: (9041,)
Channels (22): ['mag_avg_nt_mean', 'mag_avg_nt_min', 'mag_avg_nt_max', 'bx_gsm_nt_mean', 'bx_gsm_nt_min', 'bx_gsm_nt_max', 'by_gsm_nt_mean', 'by_gsm_nt_min', 'by_gsm_nt_max', 'bz_gsm_nt_mean', 'bz_gsm_nt_min', 'bz_gsm_nt_max', 'flow_speed_km_s_mean', 'flow_speed_km_s_min', 'flow_speed_km_s_max', 'proton_density_n_cc_mean', 'proton_density_n_cc_min', 'proton_density_n_cc_max', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos']
Saved outputs: [WindowsPath('work/Processed/lstm_validation_X.npy'), WindowsPath('work/Processed/lstm_validation_y.npy'), WindowsPath('work/Processed/lstm_validation_timestamps.npy'), WindowsPath('work/Processed/lstm_validation_channels.json')]

--- First sequence preview (target hour 2020-01-02T00:00:00.000000000) ---
                            mag_avg_nt_mean  mag_avg_nt_min  mag_avg_nt_max  \
hour_offset (H-24 ... H-1)          

# IV. Run pipeline — test set

In [21]:
# Load the test set
df_test_minute = pd.read_parquet(PROCESSED_DIR / 'omni_test_cleaned.parquet', columns=COLUMNS_NEEDED)
df_test_minute.shape

(1519035, 12)

In [22]:
# Drop any remaining nulls
df_test_minute = df_test_minute.dropna()

In [24]:
# Re-sample the test set to hourly
df_test_hourly = resample_to_hourly(df_test_minute)

--- resample_to_hourly: record count audit ---
Input rows (per-minute): 1,519,035
Resampled hourly buckets (2023-06-30 23:00:00 to 2026-06-30 23:00:00): 26,305
Output rows (hourly): 26,305


In [26]:
# Drop incomplete 24-hour intervals from test set
kept_test, dropped_test = drop_incomplete_intervals(df_test_hourly)

--- drop_incomplete_intervals: record count audit ---
Input hourly rows: 26,305
Candidate 3-hour boundary hours (hour % 3 == 0): 8,768
Eligible candidates with a full 24-hour history in range: 8,760
  - dropped for insufficient history (start of series): 8
  - dropped for >5 missing minutes in the lookback window: 1,202
  - dropped for a missing target kp_index: 292
Dropping 1,210 intervals (13.80%)
Keeping 7,558 intervals


In [28]:
# Build LSTM sequences for the test set
X_test, y_test, ts_test = build_lstm_sequences(
    df_test_hourly, kept_test, output_path=PROCESSED_DIR / 'lstm_test'
)

--- build_lstm_sequences: assembly audit ---
Sequences assembled: 7,558
X.shape: (7558, 24, 22)
y.shape: (7558,)
timestamps.shape: (7558,)
Channels (22): ['mag_avg_nt_mean', 'mag_avg_nt_min', 'mag_avg_nt_max', 'bx_gsm_nt_mean', 'bx_gsm_nt_min', 'bx_gsm_nt_max', 'by_gsm_nt_mean', 'by_gsm_nt_min', 'by_gsm_nt_max', 'bz_gsm_nt_mean', 'bz_gsm_nt_min', 'bz_gsm_nt_max', 'flow_speed_km_s_mean', 'flow_speed_km_s_min', 'flow_speed_km_s_max', 'proton_density_n_cc_mean', 'proton_density_n_cc_min', 'proton_density_n_cc_max', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos']
Saved outputs: [WindowsPath('work/Processed/lstm_test_X.npy'), WindowsPath('work/Processed/lstm_test_y.npy'), WindowsPath('work/Processed/lstm_test_timestamps.npy'), WindowsPath('work/Processed/lstm_test_channels.json')]

--- First sequence preview (target hour 2023-07-02T15:00:00.000000000) ---
                            mag_avg_nt_mean  mag_avg_nt_min  mag_avg_nt_max  \
hour_offset (H-24 ... H-1)                                  

# V. Verification

## 1. Confirm saved `.npy` files round-trip correctly

In [30]:
# Reload saved numpy files and confirm shapes/dtypes/values round-trip correctly
X_train_reloaded = np.load(PROCESSED_DIR / 'lstm_train_X.npy')
y_train_reloaded = np.load(PROCESSED_DIR / 'lstm_train_y.npy')
ts_train_reloaded = np.load(PROCESSED_DIR / 'lstm_train_timestamps.npy')

# Print shapes for the numpy files
print(f'X_train_reloaded: shape={X_train_reloaded.shape}, dtype={X_train_reloaded.dtype}')
print(f'y_train_reloaded: shape={y_train_reloaded.shape}, dtype={y_train_reloaded.dtype}')
print(f'ts_train_reloaded: shape={ts_train_reloaded.shape}, dtype={ts_train_reloaded.dtype}')

# Assert that reloaded numpy files match the original outputs with the LSTM sequences
assert np.array_equal(X_train_reloaded, X_train)
assert np.array_equal(y_train_reloaded, y_train)
assert np.array_equal(ts_train_reloaded, ts_train)
print('Reloaded train arrays match the in-memory arrays exactly.')

X_train_reloaded: shape=(50497, 24, 22), dtype=float32
y_train_reloaded: shape=(50497,), dtype=float32
ts_train_reloaded: shape=(50497,), dtype=datetime64[ns]
Reloaded train arrays match the in-memory arrays exactly.


In [31]:
# Reload saved numpy files and confirm shapes/dtypes/values round-trip correctly (validation set)
X_validation_reloaded = np.load(PROCESSED_DIR / 'lstm_validation_X.npy')
y_validation_reloaded = np.load(PROCESSED_DIR / 'lstm_validation_y.npy')
ts_validation_reloaded = np.load(PROCESSED_DIR / 'lstm_validation_timestamps.npy')

print(f'X_validation_reloaded: shape={X_validation_reloaded.shape}, dtype={X_validation_reloaded.dtype}')
print(f'y_validation_reloaded: shape={y_validation_reloaded.shape}, dtype={y_validation_reloaded.dtype}')
print(f'ts_validation_reloaded: shape={ts_validation_reloaded.shape}, dtype={ts_validation_reloaded.dtype}')

assert np.array_equal(X_validation_reloaded, X_validation)
assert np.array_equal(y_validation_reloaded, y_validation)
assert np.array_equal(ts_validation_reloaded, ts_validation)
print('Reloaded validation arrays match the in-memory arrays exactly.')

X_validation_reloaded: shape=(9041, 24, 22), dtype=float32
y_validation_reloaded: shape=(9041,), dtype=float32
ts_validation_reloaded: shape=(9041,), dtype=datetime64[ns]
Reloaded validation arrays match the in-memory arrays exactly.


## 2. Manual cross-check against the raw source data

Pick one kept train interval and confirm its target matches `kp_10 / 10` recomputed directly from
the per-minute parquet, and that its lookback window matches the hourly table it was sliced from.

In [32]:
sample_i = 100
sample_H = pd.Timestamp(ts_train[sample_i])

# Recompute the target directly from the raw per-minute data
expected_kp_index = df_train_minute.loc[
    df_train_minute['datetime'].between(sample_H, sample_H + pd.Timedelta(minutes=59)),
    'kp_10'
].mean() / 10.0

print(f'Target hour: {sample_H}')
print(f'kp_index from build_lstm_sequences: {y_train[sample_i]:.4f}')
print(f'kp_index recomputed from raw per-minute data: {expected_kp_index:.4f}')
assert np.isclose(y_train[sample_i], expected_kp_index, atol=1e-4)

# Confirm the lookback window strictly precedes the target hour and matches
# the hourly table it was sliced from
window_start = sample_H - pd.Timedelta(hours=24)
window_end = sample_H - pd.Timedelta(hours=1)
print(f'Lookback window: {window_start} to {window_end} (strictly before {sample_H})')

expected_window = df_train_hourly.loc[window_start:window_end, CHANNELS].to_numpy(dtype=np.float32)
assert expected_window.shape == (24, len(CHANNELS))
assert np.allclose(expected_window, X_train[sample_i])
print('Lookback window matches the source hourly data exactly.')

Target hour: 2000-01-14 12:00:00
kp_index from build_lstm_sequences: 1.7000
kp_index recomputed from raw per-minute data: 1.7000
Lookback window: 2000-01-13 12:00:00 to 2000-01-14 11:00:00 (strictly before 2000-01-14 12:00:00)
Lookback window matches the source hourly data exactly.


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=4a18bf7d-431c-4909-af8a-a54a62228a78' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>